In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
from datasets import Dataset, load_dataset, load_metric
import pandas as pd
import numpy as np

## Dataset Preparation


In [3]:
data= pd.read_json("./../dataset/ner_sent.json")
data

,Sentence_no,word_tag,only_tags,only_words,sentences
0,0,"[[বাংলাদেশে, B-LOC], [কক্সবাজার, B-LOC], [এলাক...","[B-LOC, B-LOC, O, O, O, O, O, O, O, O, O, O, O...","[বাংলাদেশে, কক্সবাজার, এলাকায়, রোহিঙ্গাদের, জ...",বাংলাদেশে কক্সবাজার এলাকায় রোহিঙ্গাদের জন্য ন...
1,1,"[[তবে, O], [টিজার, O], [দেখে, O], [দর্শকদের, O...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[তবে, টিজার, দেখে, দর্শকদের, বোঝার, উপায়, নেই,...",তবে টিজার দেখে দর্শকদের বোঝার উপায় নেই যে ছবিত...
2,2,"[[কিংবা, O], [পিতা, O], [মাতা, O], [কন্যা, O],...","[O, O, O, O, O, O, O, O, O, O]","[কিংবা, পিতা, মাতা, কন্যা, ছেলে, ভাই, বোন, স্ব...",কিংবা পিতা মাতা কন্যা ছেলে ভাই বোন স্বামী স্ত্...
3,3,"[[কিন্তু, O], [আজব, O], [ব্যাপার, O], [বাস্তবে...","[O, O, O, O, O, O]","[কিন্তু, আজব, ব্যাপার, বাস্তবে, এমনটা, হয়নি]",কিন্তু আজব ব্যাপার বাস্তবে এমনটা হয়নি
4,4,"[[বাংলাদেশও, B-LOC], [এর, O], [ব্যতিক্রম, O], ...","[B-LOC, O, O, O]","[বাংলাদেশও, এর, ব্যতিক্রম, নয়]",বাংলাদেশও এর ব্যতিক্রম নয়
...,...,...,...,...,...
22139,22139,"[[বাংলাদেশ, B-ORG], [ব্যাংকের, I-ORG], [অধীনে,...","[B-ORG, I-ORG, O, O, O, O, O, O]","[বাংলাদেশ, ব্যাংকের, অধীনে, একটি, তদন্ত, কমিটি...",বাংলাদেশ ব্যাংকের অধীনে একটি তদন্ত কমিটি কাজ করছে
22140,22140,"[[সঠিক, O], [তথ্য, O], [পাওয়া, O], [গেলে, O], ...","[O, O, O, O, O, O, O, O, O, O, O, O, B-PER, I-...","[সঠিক, তথ্য, পাওয়া, গেলে, জড়িতদের, বিরুদ্ধে, ব...",সঠিক তথ্য পাওয়া গেলে জড়িতদের বিরুদ্ধে ব্যবস্থা...
22141,22141,"[[এ, O], [কারণে, O], [বাংলাদেশ, B-LOC], [আন্তর...","[O, O, B-LOC, O, O, O, O, O, O, O, O, O]","[এ, কারণে, বাংলাদেশ, আন্তর্জাতিক, সম্প্রদায়ের,...",এ কারণে বাংলাদেশ আন্তর্জাতিক সম্প্রদায়ের কাছে ...
22142,22142,"[[সরকারের, O], [দক্ষ, O], [পরিচালনায়, O], [অর্...","[O, O, O, O, O, O, O, O, O, O]","[সরকারের, দক্ষ, পরিচালনায়, অর্থনীতির, সব, সূচক...",সরকারের দক্ষ পরিচালনায় অর্থনীতির সব সূচকে উল্ল...


#### Creating Label Encoding for NER tags

In [4]:
# ## creating mapping for the ner_tags

map_ner = {
    'O' : 0, 
    'B-PER': 1,
    'B-LOC': 2,
    'B-ORG': 3, 
    'I-PER': 4,
    'I-LOC': 5,
    'I-ORG': 6,
}

In [5]:
token_id_name = {v:k for k, v in map_ner.items()}

In [6]:
token_id_name

{0: 'O',
 1: 'B-PER',
 2: 'B-LOC',
 3: 'B-ORG',
 4: 'I-PER',
 5: 'I-LOC',
 6: 'I-ORG'}

In [7]:
import pickle 
 
with open('./../dataset/ner_token_id_name.pkl', 'wb') as f:
    pickle.dump( token_id_name ,f)

In [8]:
data['ner_tags'] = data.only_tags.apply(lambda x:list(map(map_ner.get, x)))


In [9]:
data

,Sentence_no,word_tag,only_tags,only_words,sentences,ner_tags
0,0,"[[বাংলাদেশে, B-LOC], [কক্সবাজার, B-LOC], [এলাক...","[B-LOC, B-LOC, O, O, O, O, O, O, O, O, O, O, O...","[বাংলাদেশে, কক্সবাজার, এলাকায়, রোহিঙ্গাদের, জ...",বাংলাদেশে কক্সবাজার এলাকায় রোহিঙ্গাদের জন্য ন...,"[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1,"[[তবে, O], [টিজার, O], [দেখে, O], [দর্শকদের, O...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[তবে, টিজার, দেখে, দর্শকদের, বোঝার, উপায়, নেই,...",তবে টিজার দেখে দর্শকদের বোঝার উপায় নেই যে ছবিত...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,2,"[[কিংবা, O], [পিতা, O], [মাতা, O], [কন্যা, O],...","[O, O, O, O, O, O, O, O, O, O]","[কিংবা, পিতা, মাতা, কন্যা, ছেলে, ভাই, বোন, স্ব...",কিংবা পিতা মাতা কন্যা ছেলে ভাই বোন স্বামী স্ত্...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,3,"[[কিন্তু, O], [আজব, O], [ব্যাপার, O], [বাস্তবে...","[O, O, O, O, O, O]","[কিন্তু, আজব, ব্যাপার, বাস্তবে, এমনটা, হয়নি]",কিন্তু আজব ব্যাপার বাস্তবে এমনটা হয়নি,"[0, 0, 0, 0, 0, 0]"
4,4,"[[বাংলাদেশও, B-LOC], [এর, O], [ব্যতিক্রম, O], ...","[B-LOC, O, O, O]","[বাংলাদেশও, এর, ব্যতিক্রম, নয়]",বাংলাদেশও এর ব্যতিক্রম নয়,"[2, 0, 0, 0]"
...,...,...,...,...,...,...
22139,22139,"[[বাংলাদেশ, B-ORG], [ব্যাংকের, I-ORG], [অধীনে,...","[B-ORG, I-ORG, O, O, O, O, O, O]","[বাংলাদেশ, ব্যাংকের, অধীনে, একটি, তদন্ত, কমিটি...",বাংলাদেশ ব্যাংকের অধীনে একটি তদন্ত কমিটি কাজ করছে,"[3, 6, 0, 0, 0, 0, 0, 0]"
22140,22140,"[[সঠিক, O], [তথ্য, O], [পাওয়া, O], [গেলে, O], ...","[O, O, O, O, O, O, O, O, O, O, O, O, B-PER, I-...","[সঠিক, তথ্য, পাওয়া, গেলে, জড়িতদের, বিরুদ্ধে, ব...",সঠিক তথ্য পাওয়া গেলে জড়িতদের বিরুদ্ধে ব্যবস্থা...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 0, ..."
22141,22141,"[[এ, O], [কারণে, O], [বাংলাদেশ, B-LOC], [আন্তর...","[O, O, B-LOC, O, O, O, O, O, O, O, O, O]","[এ, কারণে, বাংলাদেশ, আন্তর্জাতিক, সম্প্রদায়ের,...",এ কারণে বাংলাদেশ আন্তর্জাতিক সম্প্রদায়ের কাছে ...,"[0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
22142,22142,"[[সরকারের, O], [দক্ষ, O], [পরিচালনায়, O], [অর্...","[O, O, O, O, O, O, O, O, O, O]","[সরকারের, দক্ষ, পরিচালনায়, অর্থনীতির, সব, সূচক...",সরকারের দক্ষ পরিচালনায় অর্থনীতির সব সূচকে উল্ল...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [10]:
data['ner_tags'][3]

[0, 0, 0, 0, 0, 0]

In [11]:
data['only_tags'][3]

['O', 'O', 'O', 'O', 'O', 'O']

### Train, test and Validation Split

In [12]:
train, validate, test = np.split(data.sample(frac=1, random_state=42), [int(.8*len(data)), int(.9*len(data))])

In [13]:
train

,Sentence_no,word_tag,only_tags,only_words,sentences,ner_tags
3062,3062,"[[অং, B-PER], [সান, I-PER], [সু, I-PER], [চি, ...","[B-PER, I-PER, I-PER, I-PER, B-LOC, O, O, O, O...","[অং, সান, সু, চি, মিয়ানমারের, জনপ্রিয়, জনপ্রতি...",অং সান সু চি মিয়ানমারের জনপ্রিয় জনপ্রতিনিধি হি...,"[1, 4, 4, 4, 2, 0, 0, 0, 0, 0, 0, 0]"
3964,3964,"[[ছেলেরা, O], [মাটি, O], [কাটা, O], [শ্রমিকের,...","[O, O, O, O, O, O]","[ছেলেরা, মাটি, কাটা, শ্রমিকের, কাজ, করে]",ছেলেরা মাটি কাটা শ্রমিকের কাজ করে,"[0, 0, 0, 0, 0, 0]"
1901,1901,"[[উখিয়া, B-LOC], [থানার, O], [ভারপ্রাপ্ত, O],...","[B-LOC, O, O, O, B-PER, I-PER, I-PER, O, O, O,...","[উখিয়া, থানার, ভারপ্রাপ্ত, কর্মকর্তা, মোহাম্ম...",উখিয়া থানার ভারপ্রাপ্ত কর্মকর্তা মোহাম্মদ আবু...,"[2, 0, 0, 0, 1, 4, 4, 0, 0, 0, 0, 0, 2, 2, 2, ..."
20873,20873,"[[ওটা, O], [ছেড়ে, O], [থাকা, O], [যাবে, O], [...","[O, O, O, O, O]","[ওটা, ছেড়ে, থাকা, যাবে, না]",ওটা ছেড়ে থাকা যাবে না,"[0, 0, 0, 0, 0]"
9169,9169,"[[আগরতলা, B-LOC], [ষড়যন্ত্র, O], [মামলার, O], ...","[B-LOC, O, O, O, O, B-PER, I-PER, I-PER, I-PER...","[আগরতলা, ষড়যন্ত্র, মামলার, অন্যতম, আসামি, লেফট...",আগরতলা ষড়যন্ত্র মামলার অন্যতম আসামি লেফটেন্যান...,"[2, 0, 0, 0, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...
1051,1051,"[[আর, O], [এর, O], [ফলে, O], [কোনো, O], [সমাধা...","[O, O, O, O, O, O, O, O, O]","[আর, এর, ফলে, কোনো, সমাধান, খুঁজে, পাওয়াটাও, ক...",আর এর ফলে কোনো সমাধান খুঁজে পাওয়াটাও কঠিন হবে,"[0, 0, 0, 0, 0, 0, 0, 0, 0]"
14519,14519,"[[মহাস্থানগড়ে, B-LOC], [টুরিস্ট, O], [পুলিশ, O...","[B-LOC, O, O, O, O, O, O]","[মহাস্থানগড়ে, টুরিস্ট, পুলিশ, স্থাপনের, বিষয়টি...",মহাস্থানগড়ে টুরিস্ট পুলিশ স্থাপনের বিষয়টিও প্র...,"[2, 0, 0, 0, 0, 0, 0]"
11410,11410,"[[তিনি, O], [বলেন, O], [এই, O], [ইস্যুতে, O], ...","[O, O, O, O, O, O, O, B-LOC, O, O, O, O, O, O,...","[তিনি, বলেন, এই, ইস্যুতে, একটি, দল, ছাড়া, বাংল...",তিনি বলেন এই ইস্যুতে একটি দল ছাড়া বাংলাদেশের স...,"[0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, ..."
9221,9221,"[[স্টিভেন, B-PER], [স্মিথ, I-PER], [স্লিপে, O]...","[B-PER, I-PER, O, B-PER, I-PER, O, O, O, O, O, O]","[স্টিভেন, স্মিথ, স্লিপে, হার্দিক, পাণ্ডের, ক্য...",স্টিভেন স্মিথ স্লিপে হার্দিক পাণ্ডের ক্যাচ না ...,"[1, 4, 0, 1, 4, 0, 0, 0, 0, 0, 0]"


In [14]:
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
validate = validate.reset_index(drop=True)

### Turning to DatasetDict

In [15]:
dataset = { 
    'train' : [],
    'test' : [],
    'validation' : []
    }

In [16]:
def conv_to_dict(df, df_name):
    
    for i in range(len(df)):

        id = i
        ner_tag = df['ner_tags'][i]
        token =  df['only_words'][i]

        temp = {
            'id': str(id),
            'ner_tags': ner_tag,
            'tokens': token
        }
        

        dataset[df_name].append(temp)
        
        

In [17]:
dataset

{'train': [], 'test': [], 'validation': []}

In [18]:
conv_to_dict(train, 'train')

In [19]:
conv_to_dict(test, 'test')

In [20]:
conv_to_dict(validate, 'validation')



Followed Through https://colab.research.google.com/github/huggingface/notebooks/blob/master/examples/token_classification.ipynb#scrollTo=MKBEYOdNfWPx

In [21]:
label_list = [ 'O', 
    'B-PER',
    'B-LOC',
    'B-ORG',  
    'I-PER',
    'I-LOC',
    'I-ORG',]

In [22]:
import pickle

with open('./../dataset/ner_label_list.pkl', 'wb') as f:
    pickle.dump(label_list, f)

In [23]:
# loading labels list

import pickle
with open('./../dataset/ner_label_list.pkl', 'rb') as f:
    label_list = pickle.load(f)

In [24]:
label_list

['O', 'B-PER', 'B-LOC', 'B-ORG', 'I-PER', 'I-LOC', 'I-ORG']

### Custom DatasetDict from dataset.json format

In [25]:
import json

In [26]:
json.dump(dataset, open("./../dataset/ner_dataset_sample.json", "w")) # only run when dataset_sample.json is not available in the file system.

In [27]:
map_ner = {
     'O' : 0, 
    'B-PER': 1,
    'B-LOC': 2,
    'B-ORG': 3,  
    'I-PER': 4,
    'I-LOC': 5,
    'I-ORG': 6,
}

## All Tokens Vocabulary Preparation

In [28]:
# --- Loading train data ----

tmp = load_dataset("json", data_files={'train': "./../dataset/ner_dataset_sample.json"}, field ='train', cache_dir="./.cache")
# with tokens as string
with open('./../dataset/ner_tokens.txt', 'a+') as f:
    for i in tmp['train']:
        list_ = i["tokens"]
        
        string_token = " ".join(list_)
        f.write(string_token)
        f.write('\n')
        # print(string_token)
        # break

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-463d579a35b42838/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
# --- Loading test data ----

tmp = load_dataset("json", data_files={'train': "./../dataset/ner_dataset_sample.json"}, field ='test', cache_dir="./.cache")
# with tokens as string
with open('./../dataset/ner_tokens.txt', 'a+') as f:
    for i in tmp['train']:
        list_ = i["tokens"]
        
        string_token = " ".join(list_)
        f.write(string_token)
        f.write('\n')
        # print(string_token)
        # break

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-e057a7efc311566f/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
# --- Loading validation data ----

tmp = load_dataset("json", data_files={'train': "./../dataset/ner_dataset_sample.json"}, field ='validation', cache_dir="./.cache")
# with tokens as string
with open('./../dataset/ner_tokens.txt', 'a+') as f:
    for i in tmp['train']:
        list_ = i["tokens"]
        
        string_token = " ".join(list_)
        f.write(string_token)
        f.write('\n')
        # print(string_token)
        # break

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-1c69a4dc0778dfed/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
all_tokens = []
with open('./../dataset/ner_tokens.txt', 'r') as f:
    temp_string = f.read()
    temp_string.replace('\n', ' ')
    all_tokens = temp_string.split(' ')
    del temp_string

In [32]:
len(all_tokens)

550547

In [33]:
all_tokens = " ".join(set(all_tokens))

In [34]:
with open("./../dataset/ner_all_tokens_vocab.txt", "w") as file:
    file.write(all_tokens)